*Elvis: **[Evaluation owner]**: extracted from `airbnb_pricing_crispdm_v7.ipynb` for individual CRISP-DM phase attribution. Run from the repo root (same folder as `db.py`, `pricing_model.py`, etc.).*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Re-run training so this notebook is self-contained (doesn't depend on
# in-memory state from the Modeling notebook).
!pip install xgboost --quiet
from pricing_model import train_all_markets
results = train_all_markets()
eval_df = pd.DataFrame(results).T


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
eval_df["r2"].sort_values().plot(kind="barh", ax=ax, color="#3b82f6")
ax.set_xlabel("R² (held-out test split)")
ax.set_title("Per-market model fit — structural features only")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

**Which model family won, per market**: straight from training, no separate test needed:

In [ ]:
winner_counts = eval_df["model_type"].value_counts()
print(f"Model family selected per market:\n{winner_counts}\n")
eval_df[["model_type", "r2", "mae"]].sort_values("r2", ascending=False)


**Full metrics, including MAPE and split sizes** (MAE/RMSE in each market's own local currency; R2 and MAPE are unitless/percentage, so comparable across markets):

| Market | Currency | MAE | RMSE | R² | MAPE | n (train / test) |
|---|---|---|---|---|---|---|
| Cape Town | ZAR | 761.06 | 1,659.53 | 0.546 | 40.0% | 2,353 / 589 |
| Sydney | AUD | 85.23 | 177.77 | 0.517 | 38.8% | 2,352 / 588 |
| New York | USD | 41.78 | 67.71 | 0.482 | 33.2% | 2,358 / 590 |
| Paris | EUR | 33.24 | 57.46 | 0.398 | 32.8% | 2,363 / 591 |
| Rio de Janeiro | BRL | 284.14 | 639.33 | 0.380 | 51.0% | 2,353 / 589 |
| Bangkok | THB | 720.18 | 1,592.56 | 0.298 | 45.9% | 2,352 / 589 |
| Mexico City | MXN | 398.56 | 813.73 | 0.283 | 44.4% | 2,366 / 592 |
| Hong Kong | HKD | 252.47 | 655.52 | 0.271 | 32.0% | 2,356 / 590 |
| Rome | EUR | 33.13 | 68.53 | 0.263 | 38.9% | 2,362 / 591 |
| Istanbul | TRY | 190.60 | 360.12 | 0.228 | 49.8% | 2,362 / 591 |

Sample size per market is close to even (roughly 2,350–2,370 training rows), which rules out sample size as the explanation for the R2 spread, as the Multivariate EDA showed, the markets with stronger correlation between price and structural size features (Cape Town, Sydney, New York) are the same markets the model fits best.

**What this shows:** an earlier run with no outlier clipping and a raw-price target returned *negative* R2 on several markets (Bangkok −0.055,Hong Kong −0.227, Rio de Janeiro −0.345, Sydney −0.411): a direct, measured demonstration of why outlier handling matters on real Airbnb price data.
After the clipping + log-target fix (reproduced above), every market trains with positive R2.

**Throughput was also measured, not assumed:** naive per-listing inference ran at roughly 16ms per listing-night; batching base-price inference per market (one `model.predict()` call per market instead of one per listing, in `pricing_engine.py`) brought this to roughly 3–4.5ms per listing-night, confirming pricing should run as a scheduled batch job at real scale, not synchronously.

**What evaluation has not yet covered:** the pricing decision engine's modifier weights (event/news/ seasonality) are illustrative starting points, not backtested against real booking or revenue outcomes. The `/bookings` API endpoint captures `booked_price` specifically to enable that backtesting and future retraining, but the automated loop from booking data back into retraining has not yet been built.

### Model diagnostics: what drives price, how good are the predictions, and did we pick the right model?

The evaluation above reports MAE/RMSE/R2/MAPE per market, but three questions aren't answered by those
numbers alone: **which features actually move the price**, **how the predictions look against reality**
(not just a single R2 number), and **whether Random Forest was actually the right choice** versus
simpler or more complex alternatives. All three are answered below, reusing the exact models already
trained above (no retraining needed) plus a reconstruction of each market's held-out test split, built
with the same `random_state=42` `pricing_model.py` used, so this is the same split, not a new one.


In [ ]:
from sklearn.model_selection import train_test_split
from pricing_model import (
    _raw_listing_frame, _fill_defaults, _build_dummies,
    BASE_NUMERIC_COLUMNS, load_market_model
)
from db import get_session, Market
import numpy as np
import pandas as pd


def get_market_eval_split(city_name):
    """Reconstructs the exact held-out test split pricing_model.py used when training
    this market's model (same clipping, same random_state=42), and loads the already-
    trained model alongside it - so predictions below are from the real trained model,
    not a retrained stand-in. Also returns model_type, since different markets can now
    be served by different model families (Random Forest / Gradient Boosting / XGBoost)."""
    session = get_session()
    market = session.query(Market).filter_by(name=city_name).first()
    df = _raw_listing_frame(session, market.id)
    session.close()

    lo, hi = df["price"].quantile([0.01, 0.99])
    df = df[(df["price"] >= lo) & (df["price"] <= hi)].copy()
    y_log = np.log1p(df["price"])
    train_df, test_df, y_train_log, y_test_log = train_test_split(
        df, y_log, test_size=0.2, random_state=42
    )
    train_df, test_df = train_df.copy(), test_df.copy()

    bundle = load_market_model(market.id)
    fill_values = bundle["fill_values"]
    _fill_defaults(train_df, fill_values=fill_values)
    _fill_defaults(test_df, fill_values=fill_values)

    for d in (train_df, test_df):
        d["neighbourhood_target_enc"] = d["neighbourhood"].map(
            bundle["neighbourhood_map"]
        ).fillna(bundle["global_mean_log_price"])

    train_dummies = _build_dummies(train_df, category_lists=bundle["dummy_columns"])
    test_dummies = _build_dummies(test_df, category_lists=bundle["dummy_columns"])

    X_train = pd.concat([train_df[BASE_NUMERIC_COLUMNS].reset_index(drop=True),
                          train_dummies.reset_index(drop=True)], axis=1)[bundle["columns"]]
    X_test = pd.concat([test_df[BASE_NUMERIC_COLUMNS].reset_index(drop=True),
                         test_dummies.reset_index(drop=True)], axis=1)[bundle["columns"]]

    return {
        "market": market, "X_train": X_train, "X_test": X_test,
        "y_train_log": y_train_log.reset_index(drop=True),
        "y_test_log": y_test_log.reset_index(drop=True),
        "model": bundle["model"], "model_type": bundle.get("model_type", "random_forest"),
    }


In [ ]:
# Best- and worst-fit markets, picked dynamically from eval_df instead of hardcoded,
# since which market is best/worst no longer needs to match any earlier run.
best_market = eval_df["r2"].idxmax()
worst_market = eval_df["r2"].idxmin()
print(f"Best-fit market:  {best_market}  (R2={eval_df.loc[best_market, 'r2']}, model={eval_df.loc[best_market, 'model_type']})")
print(f"Worst-fit market: {worst_market}  (R2={eval_df.loc[worst_market, 'r2']}, model={eval_df.loc[worst_market, 'model_type']})")


#### What actually drives price

Feature importances from the trained model, for the best-fit and worst-fit markets (picked dynamically
above, not hardcoded) - so the comparison also shows whether the *same* features matter in a market
the model explains well versus one it explains poorly. Titles show which model family actually won
each market - this may not be Random Forest anymore.


In [ ]:
import matplotlib.pyplot as plt
from db import get_session, Market
from pricing_model import load_market_model

session = get_session()
for city_name in [best_market, worst_market]:
    market = session.query(Market).filter_by(name=city_name).first()
    bundle = load_market_model(market.id)
    importances = pd.Series(bundle["model"].feature_importances_, index=bundle["columns"]).sort_values()

    fig, ax = plt.subplots(figsize=(7, 5))
    importances.tail(12).plot(kind="barh", ax=ax, color="#3b82f6")
    ax.set_title(f"What drives price — {city_name} ({bundle.get('model_type', 'random_forest')}, "
                 f"R2={eval_df.loc[city_name, 'r2']})")
    ax.set_xlabel("Feature importance")
    plt.tight_layout()
    plt.show()
session.close()


#### Predicted vs. actual price, best- and worst-fit markets

A single R2 number hides *how* a model is wrong. Plotting predictions against actual held-out prices
shows it directly: points on the red dashed line are exact predictions, and the spread above/below it
is the error the MAE/RMSE numbers summarize. Each title shows the model family that actually won that
market.


In [ ]:
for city_name in [best_market, worst_market]:
    split = get_market_eval_split(city_name)
    y_test = np.expm1(split["y_test_log"])
    preds = np.expm1(split["model"].predict(split["X_test"]))

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y_test, preds, alpha=0.3, s=10, color="#3b82f6")
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    ax.plot(lims, lims, "r--", linewidth=1, label="Perfect prediction")
    ax.set_xlabel(f"Actual price ({split['market'].currency})")
    ax.set_ylabel(f"Predicted price ({split['market'].currency})")
    ax.set_title(f"Predicted vs. actual — {city_name} ({split['model_type']})")
    ax.legend()
    plt.tight_layout()
    plt.show()


#### Which model actually won, across all 10 markets?

This used to be a separate notebook exercise: retrain fresh Gradient Boosting/XGBoost models here and
compare them against the saved Random Forest. That's no longer necessary — `pricing_model.py` now tunes
all three candidates **during training itself** and keeps whichever wins on CV score, so `eval_df`
(built directly from `train_all_markets()`'s `results`) already contains `model_type` (the winner) and
`candidate_cv_r2` (all three CV scores) for every market. This just visualizes what training already
decided, using the real training-time CV scores rather than a separate, less rigorous test-set
comparison.


In [ ]:
winner_counts = eval_df["model_type"].value_counts()
print(f"Random Forest is the best model in {winner_counts.get('random_forest', 0)} of {len(eval_df)} markets.")
print(f"Winner counts:\n{winner_counts}\n")

cv_scores_df = pd.DataFrame(eval_df["candidate_cv_r2"].to_dict()).T
fig, ax = plt.subplots(figsize=(9, 5))
cv_scores_df.plot(kind="bar", ax=ax, color=["#1d4ed8", "#93c5fd", "#f97316"][:cv_scores_df.shape[1]])
ax.set_ylabel("CV R² (train-only — this is what selected the winner)")
ax.set_title("Per-market candidate CV scores")
ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

eval_df[["model_type", "r2", "mae", "candidate_cv_r2"]].sort_values("r2", ascending=False)


In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# Naive baseline: predict each market's own training-fold mean price, every time.
# Scored with the same 5-fold CV structure RandomizedSearchCV used for the tuned
# candidates (shuffle=False - sklearn's default when cv=5 is passed as an int),
# so this is a fair apples-to-apples comparison, not a different kind of test.
naive_cv_r2 = {}
for city_name in eval_df.index:
    split = get_market_eval_split(city_name)
    y_train_log = split["y_train_log"].values
    kf = KFold(n_splits=5, shuffle=False)
    fold_scores = []
    for tr_idx, val_idx in kf.split(y_train_log):
        fold_mean_price = np.expm1(y_train_log[tr_idx]).mean()
        y_val = np.expm1(y_train_log[val_idx])
        preds = np.full_like(y_val, fold_mean_price)
        fold_scores.append(r2_score(y_val, preds))
    naive_cv_r2[city_name] = np.mean(fold_scores)

# Pull the three tuned candidates' CV scores (already computed during training)
# and line all four up together.
chart_df = pd.DataFrame({
    "Naive (market mean)": pd.Series(naive_cv_r2),
    "Random Forest": eval_df["candidate_cv_r2"].apply(lambda d: d.get("random_forest")),
    "Gradient Boosting": eval_df["candidate_cv_r2"].apply(lambda d: d.get("gradient_boosting")),
    "XGBoost": eval_df["candidate_cv_r2"].apply(lambda d: d.get("xgboost")),
}).sort_values("XGBoost", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
chart_df.plot(kind="bar", ax=ax, color=["#9ca3af", "#1d4ed8", "#93c5fd", "#f97316"])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("CV R² (train-only)")
ax.set_title("Naive baseline vs. tuned candidates, per market")
ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

chart_df.round(3)